<a href="https://colab.research.google.com/github/ODUNAYOMIDE-YAKUBU/sentiment_chatbot/blob/main/phase4_dialogue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install transformers torch pandas -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil
import os

# Copy empathetic_dialogues.csv from Drive if you saved it there
shutil.copy(
    '/content/drive/MyDrive/sentiment_chatbot/empathetic_dialogues.csv',
    'empathetic_dialogues.csv'
)
print("Files ready!")

Files ready!


In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class BertLSTMClassifier(nn.Module):
    def __init__(self, bert_model_name="bert-base-uncased",
                 hidden_dim=256, num_layers=2,
                 num_classes=3, dropout=0.3):
        super(BertLSTMClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        bert_out        = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = bert_out.last_hidden_state
        lstm_out, _     = self.lstm(sequence_output)
        lstm_final      = lstm_out[:, -1, :]
        out             = self.dropout(lstm_final)
        return self.classifier(out)

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Load model weights directly from Google Drive
model_path = '/content/drive/MyDrive/sentiment_chatbot/models/bert_lstm_best_v2.pt'
sentiment_model = BertLSTMClassifier().to(device)
sentiment_model.load_state_dict(
    torch.load(model_path, map_location=device)
)
sentiment_model.eval()
print("Sentiment model loaded from Drive successfully!")
print(f"Running on: {device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentiment model loaded from Drive successfully!
Running on: cuda


In [ ]:
def predict_sentiment(text):
    encoding = tokenizer(
        text,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        logits = sentiment_model(input_ids, attention_mask)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred   = torch.argmax(logits, dim=1).item()

    label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
    return {
        'sentiment': label_map[pred],
        'confidence': float(probs[pred]),
        'scores': {
            'negative': float(probs[0]),
            'neutral':  float(probs[1]),
            'positive': float(probs[2])
        }
    }

# Test it
test_sentences = [
    "I am so happy today abeg this thing don work!",
    "I dey very angry right now nobody help me",
    "I go send you the details later today",
    "This life no balance at all everything dey hard",
    "Thank God the result come out well"
]

for s in test_sentences:
    result = predict_sentiment(s)
    print(f"Text:       {s}")
    print(f"Sentiment:  {result['sentiment']} ({result['confidence']:.2%} confidence)")
    print(f"Scores:     {result['scores']}")
    print()

Text:       I am so happy today abeg this thing don work!
Sentiment:  positive (94.79% confidence)
Scores:     {'negative': 0.05165424942970276, 'neutral': 0.0004773675464093685, 'positive': 0.947868287563324}

Text:       I dey very angry right now nobody help me
Sentiment:  positive (85.01% confidence)
Scores:     {'negative': 0.14881619811058044, 'neutral': 0.0010429512476548553, 'positive': 0.8501408696174622}

Text:       I go send you the details later today
Sentiment:  neutral (55.95% confidence)
Scores:     {'negative': 0.0818130150437355, 'neutral': 0.559468150138855, 'positive': 0.3587188720703125}

Text:       This life no balance at all everything dey hard
Sentiment:  positive (56.84% confidence)
Scores:     {'negative': 0.42511576414108276, 'neutral': 0.006495344452559948, 'positive': 0.5683888792991638}

Text:       Thank God the result come out well
Sentiment:  positive (98.30% confidence)
Scores:     {'negative': 0.016682889312505722, 'neutral': 0.0002959198609460145, '

In [ ]:
class DialogueStateTracker:
    def __init__(self):
        self.history          = []   # stores (user_text, sentiment, response)
        self.sentiment_trail  = []   # tracks sentiment over turns
        self.turn_count       = 0

    def update(self, user_text, sentiment, response):
        self.history.append({
            'turn':      self.turn_count,
            'user':      user_text,
            'sentiment': sentiment,
            'response':  response
        })
        self.sentiment_trail.append(sentiment)
        self.turn_count += 1

    def get_dominant_sentiment(self):
        if not self.sentiment_trail:
            return 'neutral'
        return max(set(self.sentiment_trail), key=self.sentiment_trail.count)

    def is_escalating_negative(self):
        # Check if last 2 turns were both negative
        if len(self.sentiment_trail) >= 2:
            return all(s == 'negative' for s in self.sentiment_trail[-2:])
        return False

    def get_context_summary(self):
        return {
            'turns':              self.turn_count,
            'dominant_sentiment': self.get_dominant_sentiment(),
            'escalating':         self.is_escalating_negative(),
            'last_sentiment':     self.sentiment_trail[-1] if self.sentiment_trail else None
        }

    def reset(self):
        self.history         = []
        self.sentiment_trail = []
        self.turn_count      = 0

tracker = DialogueStateTracker()
print("Dialogue state tracker ready!")

Dialogue state tracker ready!


In [ ]:
import random

RESPONSES = {
    'positive': {
        'default': [
            "That's wonderful to hear! I'm glad things are going well for you. 😊",
            "Fantastic! Keep that positive energy going — you deserve it!",
            "That's really great news! What else is making your day good?",
            "I love hearing that! Your positivity is contagious. 🌟",
            "Amazing! It sounds like things are really working out for you."
        ],
        'follow_up': [
            "You seem to be on a great streak! What's been the highlight?",
            "That positive energy is building up nicely! Tell me more.",
            "You're really thriving! I'm happy to hear that. 🎉"
        ]
    },
    'negative': {
        'default': [
            "I'm really sorry to hear that. That sounds genuinely tough. 💙",
            "That must be really hard for you. I hear you and I'm here.",
            "I understand — it's okay to feel this way. You're not alone.",
            "That sounds frustrating. Would you like to talk about it more?",
            "I'm sorry you're going through this. How can I support you right now?"
        ],
        'escalating': [
            "I can see this has been really weighing on you. Please know that you matter and things can get better. 💙",
            "It sounds like things have been really difficult lately. Have you been able to talk to someone you trust?",
            "I'm genuinely concerned and I want you to know you don't have to face this alone. 🤝"
        ],
        'nigerian_context': [
            "E go better, I promise you. This kind situation no go last forever. 💪",
            "Abeg no let it weigh you down too much — you strong pass this thing.",
            "Na so life be sometimes, but you go scale through. I dey here for you."
        ]
    },
    'neutral': {
        'default': [
            "I see, thanks for sharing that with me. Tell me more.",
            "Alright, I understand. How are you feeling about it overall?",
            "Got it. Is there anything specific on your mind right now?",
            "Okay, I'm following you. What would you like to talk about?",
            "I hear you. Feel free to share whatever is on your mind."
        ]
    }
}

def get_response(sentiment, context):
    if sentiment == 'negative' and context['escalating']:
        pool = RESPONSES['negative']['escalating']
    elif sentiment == 'negative':
        pool = RESPONSES['negative']['default']
    elif sentiment == 'positive' and context['turns'] > 1:
        pool = RESPONSES['positive']['follow_up']
    elif sentiment == 'positive':
        pool = RESPONSES['positive']['default']
    else:
        pool = RESPONSES['neutral']['default']

    return random.choice(pool)

print("Response engine ready!")

Response engine ready!


In [ ]:
class SentimentAwareDialogueManager:
    def __init__(self, sentiment_predictor, response_engine, tracker):
        self.predict  = sentiment_predictor
        self.respond  = response_engine
        self.tracker  = tracker

    def process(self, user_input):
        # Step 1: Detect sentiment
        result    = self.predict(user_input)
        sentiment = result['sentiment']
        confidence= result['confidence']

        # Step 2: Get dialogue context
        context = self.tracker.get_context_summary()

        # Step 3: Generate appropriate response
        response = self.respond(sentiment, context)

        # Step 4: Update dialogue state
        self.tracker.update(user_input, sentiment, response)

        return {
            'user_input':  user_input,
            'sentiment':   sentiment,
            'confidence':  confidence,
            'response':    response,
            'context':     context
        }

    def reset(self):
        self.tracker.reset()
        print("Conversation reset.")

dialogue_manager = SentimentAwareDialogueManager(
    sentiment_predictor=predict_sentiment,
    response_engine=get_response,
    tracker=tracker
)
print("Dialogue manager ready!")

Dialogue manager ready!


In [ ]:
dialogue_manager.reset()

test_conversation = [
    "I just got promoted at work today!",
    "But my colleague has been making things difficult for me",
    "Nobody even acknowledged my hard work except my boss",
    "I don't know if I should confront him or just ignore it",
    "I go just focus on my work sha"
]

print("=" * 60)
print("SENTIMENT-AWARE CONVERSATION TEST")
print("=" * 60)

for user_msg in test_conversation:
    result = dialogue_manager.process(user_msg)
    print(f"\nUser      : {result['user_input']}")
    print(f"Sentiment : {result['sentiment'].upper()} ({result['confidence']:.1%})")
    print(f"Bot       : {result['response']}")
    print(f"Context   : {result['context']}")
    print("-" * 60)

Conversation reset.
SENTIMENT-AWARE CONVERSATION TEST

User      : I just got promoted at work today!
Sentiment : NEGATIVE (98.2%)
Bot       : I understand — it's okay to feel this way. You're not alone.
Context   : {'turns': 0, 'dominant_sentiment': 'neutral', 'escalating': False, 'last_sentiment': None}
------------------------------------------------------------

User      : But my colleague has been making things difficult for me
Sentiment : NEGATIVE (97.7%)
Bot       : I'm really sorry to hear that. That sounds genuinely tough. 💙
Context   : {'turns': 1, 'dominant_sentiment': 'negative', 'escalating': False, 'last_sentiment': 'negative'}
------------------------------------------------------------

User      : Nobody even acknowledged my hard work except my boss
Sentiment : NEGATIVE (94.2%)
Bot       : It sounds like things have been really difficult lately. Have you been able to talk to someone you trust?
Context   : {'turns': 2, 'dominant_sentiment': 'negative', 'escalating': Tr

In [ ]:
import json

# Save response templates for use in the app
with open("response_templates.json", "w") as f:
    json.dump(RESPONSES, f, indent=2)

print("Response templates saved!")

from google.colab import files
files.download("response_templates.json")